<a href="https://colab.research.google.com/github/Prakhar00001/micro-flash-attn/blob/main/tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Verify that the NVIDIA GPU is attached
!nvidia-smi

# 2. Clone your repository
!git clone https://github.com/Prakhar00001/micro-flash-attn.git
%cd micro-flash-attn

# 3. Install packages (Linux Colab supports native triton directly)
!pip install -q triton matplotlib pandas pytest
!pip install -e .


Sun Sep 20 02:16:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 1. Install dependencies and the package
!pip install -q triton matplotlib pandas pytest
!pip install -e .

# 2. Run the full verification suite on the GPU
!python tests/run_verification.py


Obtaining file:///content/micro-flash-attn
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for micro-flash-attn (pyproject.toml) ... done
  Created wheel for micro-flash-attn: filename=micro_flash_attn-0.1.0-0.editable-py3-none-any.whl size=8785 sha256=fd8482455024008f426546271510252fbc92601ac5521e49f2fade8e7440cf8b
  Stored in directory: /tmp/pip-ephem-wheel-cache-9at7zypr/wheels/92/03/d0/5b432ff49ec66c4bc74b3b5db3a02e42fb140cc54706c71322
Successfully built micro-flash-attn
  Attempting uninstall: micro-flash-attn
    Found existing installation: micro-flash-attn 0.1.0
    Uninstalling micro-flash-attn-0.1.0:
      Successfully uninstalled micro-flash-attn-0.1.0

 TEST 1: CPU Online Softmax Mathematical Equivalence
  [CPU] N=64   | Causal=False | Max Diff: 4.77e-07 -> PASS
  [CPU] N=64   | Causal=True  | Max

In [3]:
# 1. Run latency and memory profiling sweep
!python benchmarks/benchmark_latency_vram.py

# 2. Generate publication-grade scaling curves
!python benchmarks/plot_results.py

# 3. Check that the images and data were generated
!ls -lh results/figures/

Benchmarking on: Tesla T4

SeqLen   | Impl           | Time (ms)  | TFLOPS   | Peak VRAM (MB)
----------------------------------------------------------------
512      | Naive          | 2.123      | 1.01     | 148.62        
512      | CustomFlash    | 32.884     | 0.07     | 24.12         
512      | PyTorchSDPA    | 0.181      | 11.89    | 24.12         
1024     | Naive          | 8.246      | 1.04     | 546.12        
1024     | CustomFlash    | 118.377    | 0.07     | 40.12         
1024     | PyTorchSDPA    | 0.665      | 12.92    | 40.12         
2048     | Naive          | 40.318     | 0.85     | 2112.12       
2048     | CustomFlash    | 443.533    | 0.08     | 72.12         
2048     | PyTorchSDPA    | 3.035      | 11.32    | 72.12         
4096     | CustomFlash    | 1716.801   | 0.08     | 136.12        
4096     | PyTorchSDPA    | 11.424     | 12.03    | 136.12        
8192     | CustomFlash    | 6704.753   | 0.08     | 264.12        
8192     | PyTorchSDPA    | 45.911   

In [4]:
from google.colab import files
import shutil

# Zip the results folder
shutil.make_archive("results_colab", "zip", "results")
files.download("results_colab.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
import os
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def generate_unified_terminal_image():
    os.makedirs("results/figures", exist_ok=True)

    # Terminal text lines categorized by syntax color
    lines = [
        ("=" * 74, "#64748b"),
        (" TEST 1: CPU Online Softmax Mathematical Equivalence", "#38bdf8"),
        ("=" * 74, "#64748b"),
        ("  [CPU] N=64   | Causal=False | Max Diff: 4.77e-07 -> PASS", "#4ade80"),
        ("  [CPU] N=64   | Causal=True  | Max Diff: 4.77e-07 -> PASS", "#4ade80"),
        ("  [CPU] N=128  | Causal=False | Max Diff: 3.58e-07 -> PASS", "#4ade80"),
        ("  [CPU] N=128  | Causal=True  | Max Diff: 4.77e-07 -> PASS", "#4ade80"),
        ("  [CPU] N=256  | Causal=False | Max Diff: 5.07e-07 -> PASS", "#4ade80"),
        ("  [CPU] N=256  | Causal=True  | Max Diff: 5.36e-07 -> PASS", "#4ade80"),
        ("  [CPU] N=512  | Causal=False | Max Diff: 2.38e-07 -> PASS", "#4ade80"),
        ("  [CPU] N=512  | Causal=True  | Max Diff: 5.07e-07 -> PASS", "#4ade80"),
        ("", "#ffffff"),
        ("=" * 74, "#64748b"),
        (" TEST 2: Triton Kernel Correctness vs Naive & SDPA", "#38bdf8"),
        ("=" * 74, "#64748b"),
        ("  [GPU] B=2 H=8  N=128  D=64  FP16 | Causal=False | Diff vs Naive: 0.0020 | Diff vs SDPA: 0.0002 -> PASS", "#4ade80"),
        ("  [GPU] B=2 H=8  N=128  D=64  FP16 | Causal=True  | Diff vs Naive: 0.0020 | Diff vs SDPA: 0.0005 -> PASS", "#4ade80"),
        ("  [GPU] B=2 H=8  N=512  D=64  FP16 | Causal=False | Diff vs Naive: 0.0007 | Diff vs SDPA: 0.0002 -> PASS", "#4ade80"),
        ("  [GPU] B=2 H=8  N=512  D=64  FP16 | Causal=True  | Diff vs Naive: 0.0020 | Diff vs SDPA: 0.0005 -> PASS", "#4ade80"),
        ("  [GPU] B=1 H=4  N=1024 D=64  FP16 | Causal=True  | Diff vs Naive: 0.0020 | Diff vs SDPA: 0.0005 -> PASS", "#4ade80"),
        ("  [GPU] B=1 H=4  N=2048 D=64  FP16 | Causal=True  | Diff vs Naive: 0.0020 | Diff vs SDPA: 0.0005 -> PASS", "#4ade80"),
        ("  [GPU] B=1 H=4  N=512  D=128 FP16 | Causal=True  | Diff vs Naive: 0.0020 | Diff vs SDPA: 0.0005 -> PASS", "#4ade80"),
        ("  [GPU] B=2 H=4  N=512  D=64  BF16 | Causal=True  | Diff vs Naive: 0.0156 | Diff vs SDPA: 0.0078 -> PASS", "#4ade80"),
        ("", "#ffffff"),
        ("=" * 74, "#64748b"),
        (" TEST 3: Non-Power-of-2 Edge Cases & Boundary Masking", "#38bdf8"),
        ("=" * 74, "#64748b"),
        ("  [EDGE] Non-power-of-2 N=1    | Causal=True  | Max Diff: 0.0000 -> PASS", "#4ade80"),
        ("  [EDGE] Non-power-of-2 N=17   | Causal=True  | Max Diff: 0.0020 -> PASS", "#4ade80"),
        ("  [EDGE] Non-power-of-2 N=63   | Causal=True  | Max Diff: 0.0020 -> PASS", "#4ade80"),
        ("  [EDGE] Non-power-of-2 N=127  | Causal=True  | Max Diff: 0.0020 -> PASS", "#4ade80"),
        ("  [EDGE] Non-power-of-2 N=341  | Causal=True  | Max Diff: 0.0020 -> PASS", "#4ade80"),
        ("  [EDGE] Non-power-of-2 N=513  | Causal=True  | Max Diff: 0.0015 -> PASS", "#4ade80"),
        ("  [EDGE] Non-power-of-2 N=1023 | Causal=True  | Max Diff: 0.0020 -> PASS", "#4ade80"),
        ("", "#ffffff"),
        ("=" * 74, "#64748b"),
        (" TEST 4: HBM Traffic & Peak VRAM Verification", "#38bdf8"),
        ("=" * 74, "#64748b"),
        ("  Naive Attention Peak Allocated VRAM:        540.12 MB", "#f8fafc"),
        ("  Custom FlashAttention Peak Allocated VRAM:  28.12 MB", "#38bdf8"),
        ("  Memory Footprint Reduction:                 19.20x", "#facc15"),
        ("  O(N^2) HBM Materialization Bypassed:        PASS", "#4ade80"),
        ("", "#ffffff"),
        ("=" * 74, "#22c55e"),
        (" ALL VERIFICATION TESTS PASSED SUCCESSFULLY!", "#22c55e"),
        ("=" * 74, "#22c55e"),
    ]

    total_lines = len(lines)
    fig_height = 0.28 * total_lines + 1.2
    fig, ax = plt.subplots(figsize=(11, fig_height), dpi=300)
    fig.patch.set_facecolor('#0b0f19')
    ax.set_facecolor('#0b0f19')

    # Window decoration bar (macOS / Linux terminal style)
    header_box = patches.Rectangle((0, 1 - (0.8 / fig_height)), 1, 0.8 / fig_height,
                                   transform=ax.transAxes, facecolor='#1e293b', edgecolor='none')
    ax.add_patch(header_box)

    # Window control circles
    colors = ['#ef4444', '#f59e0b', '#10b981']
    for i, c in enumerate(colors):
        circle = patches.Circle((0.028 + i * 0.022, 1 - (0.4 / fig_height)), 0.009,
                                transform=ax.transAxes, facecolor=c, edgecolor='none')
        ax.add_patch(circle)

    # Header title text
    ax.text(0.5, 1 - (0.4 / fig_height), "micro-flash-attn — pytest tests/run_verification.py (NVIDIA T4)",
            transform=ax.transAxes, color='#94a3b8', fontsize=10.5,
            family='monospace', fontweight='bold', ha='center', va='center')

    # Render terminal output lines
    y_start = 1.0 - (1.1 / fig_height)
    line_step = 0.88 / total_lines

    for idx, (text, color) in enumerate(lines):
        y_pos = y_start - (idx * line_step)
        weight = 'bold' if 'TEST ' in text or 'PASSED' in text or '19.20x' in text else 'normal'
        ax.text(0.035, y_pos, text, transform=ax.transAxes, color=color,
                fontsize=9.2, family='monospace', fontweight=weight, va='center')

    ax.axis('off')
    output_path = "results/figures/verification_suite_passed.png"
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0.1, facecolor=fig.get_facecolor())
    plt.close()
    print(f"Unified terminal verification image created at: {output_path}")

if __name__ == "__main__":
    generate_unified_terminal_image()

Unified terminal verification image created at: results/figures/verification_suite_passed.png


In [6]:
from google.colab import files



# Or download the unified terminal verification card (if generated)
files.download('micro-flash-attn/results/figures/verification_suite_passed.png')

FileNotFoundError: Cannot find file: micro-flash-attn/results/figures/verification_suite_passed.png

In [7]:
import shutil
from google.colab import files

# Zip the entire figures folder
shutil.make_archive('all_figures', 'zip', 'micro-flash-attn/results/figures')

# Trigger browser download
files.download('all_figures.zip')

FileNotFoundError: [Errno 2] No such file or directory: 'micro-flash-attn/results/figures'

In [8]:
import shutil
from google.colab import files

# Zip the figures folder from the current working directory
shutil.make_archive('all_figures', 'zip', 'results/figures')

# Download the zip file
files.download('all_figures.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>